# Controle rapide — comparateur avec/sans vignette

Lit `output/trajets.json` (genere par `python -m pipeline.build_dataset`) et affiche un tableau de controle par destination : distance, temps, surcout, surconsommation, CO2. Relancer cette cellule apres chaque regeneration du pipeline.


In [ ]:
import json
from pathlib import Path
import pandas as pd

TRAJETS_JSON = Path("output/trajets.json")
data = json.loads(TRAJETS_JSON.read_text(encoding="utf-8"))
print(f"{len(data)} destinations chargees")

## Tableau de synthese

In [ ]:
rows = []
for d in data:
    rows.append({
        "destination": d["nom"],
        "km (sans vignette)": d["sans"]["distance_km"],
        "min (sans vignette)": d["sans"]["temps_min"],
        "km (avec vignette)": d["avec"]["distance_km"],
        "min (avec vignette)": d["avec"]["temps_min"],
        "delta km": d["delta"]["delta_km"],
        "delta min": d["delta"]["delta_temps_min"],
        "delta carburant (L)": d["delta"]["delta_carburant_l"],
        "delta cout (EUR)": d["delta"]["delta_cout_eur"],
        "delta CO2 (kg)": d["delta"]["delta_co2_kg"],
        "cassures detectees": d["sans"]["nb_cassures_detectees"],
    })

df = pd.DataFrame(rows).sort_values("delta km", ascending=False).reset_index(drop=True)
df.style.format({
    "km (sans vignette)": "{:.1f}", "km (avec vignette)": "{:.1f}", "delta km": "{:+.1f}",
    "min (sans vignette)": "{:.0f}", "min (avec vignette)": "{:.0f}", "delta min": "{:+.0f}",
    "delta carburant (L)": "{:+.1f}", "delta cout (EUR)": "{:+.2f}", "delta CO2 (kg)": "{:+.1f}",
}).background_gradient(subset=["delta km"], cmap="Oranges")

## Alertes a revoir manuellement

Destinations avec au moins une "cassure" detectee dans le trace sans-vignette (portion sans court chemin routable trouve -- a revisualiser dans gpx.studio en priorite, cf. README section "Itineraire sans vignette").

In [ ]:
for d in data:
    if d["sans"]["nb_cassures_detectees"]:
        print(f"- {d['nom']} ({d['id']}) : {d['sans']['nb_cassures_detectees']} cassure(s) -- voir output/stats/{d['id']}.json")

## Statistiques editoriales (analyse)

In [ ]:
analyse_rows = []
for d in data:
    a = d["analyse"]
    analyse_rows.append({
        "destination": d["nom"],
        "communes": a["nb_communes_traversees"],
        "villages": a["nb_villages_traverses"],
        "giratoires": a["nb_giratoires"],
        "changements de route": a["nb_changements_de_route"],
        "feux": a["nb_feux"],
        "vitesse moy. (km/h)": a["vitesse_moyenne_kmh"],
        "vitesse med. (km/h)": a["vitesse_mediane_kmh"],
        "km < 50 km/h": a["temps_moins_50kmh_min"],
        "km > 90 km/h": a["temps_plus_90kmh_min"],
    })
pd.DataFrame(analyse_rows)

## Repartition par type de route (sans vignette)

In [ ]:
road_rows = []
for d in data:
    a = d["analyse"]
    road_rows.append({
        "destination": d["nom"],
        "communale (km)": a["distance_communale_km"],
        "departementale (km)": a["distance_departementale_km"],
        "nationale/regionale (km)": a["distance_nationale_regionale_km"],
        "autoroute (km)": a["distance_autoroute_km"],
        "inconnu (km)": a["distance_type_inconnu_km"],
    })
pd.DataFrame(road_rows)

## Graphique rapide : surcout km / temps par destination

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_sorted = df.sort_values("delta km")
axes[0].barh(df_sorted["destination"], df_sorted["delta km"], color="#d95f02")
axes[0].set_title("Kilometres supplementaires (sans vs avec vignette)")
axes[0].set_xlabel("km")

df_sorted2 = df.sort_values("delta min")
axes[1].barh(df_sorted2["destination"], df_sorted2["delta min"], color="#7570b3")
axes[1].set_title("Minutes supplementaires")
axes[1].set_xlabel("min")
plt.tight_layout()
plt.show()